# DSPy Usage Tracking

This notebook demonstrates DSPy 3.x's built-in usage tracking for monitoring token usage and costs.

**What You'll Learn:**
- How to enable usage tracking in DSPy
- How to retrieve and analyze usage statistics
- How to track usage per module
- How to optimize based on usage data
- How to implement cost monitoring

## Setup

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import (
    setup_default_lm, configure_dspy,
    get_usage_stats, print_usage_stats,
    print_step, print_result, print_error
)
from dotenv import load_dotenv

load_dotenv('../../.env')

## Part 1: Enabling Usage Tracking

Enable usage tracking by passing `track_usage=True` to `configure_dspy()`. DSPy will then automatically track all LM calls.

In [ ]:
try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=500)
    configure_dspy(lm=lm, track_usage=True)
    print_result("Language model configured with usage tracking enabled!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")

print("\nUsage tracking is now active!")
print("DSPy will automatically track:")
print("  - Total tokens used (input + output)")
print("  - Input tokens")
print("  - Output tokens")
print("  - Number of API calls")
print("  - Cost estimates (when available)")

## Part 2: Basic Usage Tracking

Track usage for simple operations and review the statistics.

In [ ]:
class QuestionAnswering(dspy.Signature):
    """Answer questions concisely."""
    question = dspy.InputField()
    answer = dspy.OutputField()

qa = dspy.Predict(QuestionAnswering)

questions = [
    "What is the capital of France?",
    "Who wrote Romeo and Juliet?",
    "What is 2 + 2?"
]

print("Making predictions...")
for i, q in enumerate(questions, 1):
    result = qa(question=q)
    print(f"  {i}. Q: {q}")
    print(f"     A: {result.answer}")

In [ ]:
# Display usage statistics
print("Usage statistics after 3 predictions:")
print_usage_stats()

usage = get_usage_stats()
if usage:
    input_cost_per_1k = 0.005
    output_cost_per_1k = 0.015

    if 'input_tokens' in usage and 'output_tokens' in usage:
        total_cost = (
            (usage['input_tokens'] / 1000) * input_cost_per_1k +
            (usage['output_tokens'] / 1000) * output_cost_per_1k
        )
        print(f"Estimated cost: ${total_cost:.4f}")

## Part 3: Module-Level Usage Tracking

Track usage for custom multi-step modules by measuring deltas before and after operations.

In [ ]:
class SmartQA(dspy.Module):
    """A multi-step QA system with usage tracking."""
    def __init__(self):
        super().__init__()

        class ClassifyQuestion(dspy.Signature):
            """Classify the question type."""
            question = dspy.InputField()
            question_type = dspy.OutputField(desc="factual, mathematical, or creative")

        class AnswerQuestion(dspy.Signature):
            """Answer based on question type."""
            question = dspy.InputField()
            question_type = dspy.InputField()
            answer = dspy.OutputField()

        self.classifier = dspy.Predict(ClassifyQuestion)
        self.answerer = dspy.ChainOfThought(AnswerQuestion)

    def forward(self, question):
        initial_usage = get_usage_stats()
        classification = self.classifier(question=question)
        answer = self.answerer(question=question, question_type=classification.question_type)
        final_usage = get_usage_stats()

        usage_delta = {}
        if initial_usage and final_usage:
            for key in final_usage:
                if key in initial_usage:
                    usage_delta[key] = final_usage[key] - initial_usage[key]

        return dspy.Prediction(
            question_type=classification.question_type,
            answer=answer.answer,
            reasoning=answer.reasoning,
            usage=usage_delta
        )

smart_qa = SmartQA()

test_question = "If I have 10 apples and give away 3, how many do I have left?"
result = smart_qa(question=test_question)

print(f"Question: {test_question}")
print(f"Type: {result.question_type}")
print(f"Reasoning: {result.reasoning}")
print(f"Answer: {result.answer}")

if result.usage:
    print(f"\nUsage for this module call:")
    for key, value in result.usage.items():
        print(f"  {key}: {value}")

## Part 4: Comparing Approaches

Use tracking to compare the efficiency of Predict vs ChainOfThought.

In [ ]:
class SimpleSignature(dspy.Signature):
    """Solve a problem."""
    problem = dspy.InputField()
    solution = dspy.OutputField()

# Approach 1: Simple Predict
print("Approach 1: Using dspy.Predict")
simple_predictor = dspy.Predict(SimpleSignature)

usage_before = get_usage_stats()
result1 = simple_predictor(problem="What is 7 * 8?")
usage_after = get_usage_stats()

if usage_before and usage_after:
    tokens_used = usage_after.get('total_tokens', 0) - usage_before.get('total_tokens', 0)
    print(f"  Tokens used: ~{tokens_used}")
print(f"  Solution: {result1.solution}")

# Approach 2: ChainOfThought
print("\nApproach 2: Using dspy.ChainOfThought")
cot_predictor = dspy.ChainOfThought(SimpleSignature)

usage_before = get_usage_stats()
result2 = cot_predictor(problem="What is 7 * 8?")
usage_after = get_usage_stats()

if usage_before and usage_after:
    tokens_used = usage_after.get('total_tokens', 0) - usage_before.get('total_tokens', 0)
    print(f"  Tokens used: ~{tokens_used}")
print(f"  Solution: {result2.solution}")

print("\nKey insight:")
print("ChainOfThought typically uses more tokens (includes reasoning)")
print("but may provide better quality answers for complex problems.")

## Part 5: Cost Monitoring

Implement a cost monitor with budget limits and alerts.

In [ ]:
class CostMonitor:
    """Monitor and track LM costs."""

    def __init__(self, model="gpt-4o"):
        self.pricing = {
            "gpt-4o": {"input": 0.005, "output": 0.015},
            "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
            "gpt-3.5-turbo": {"input": 0.0005, "output": 0.0015},
            "claude-3-7-sonnet": {"input": 0.003, "output": 0.015},
        }
        self.model = model
        self.initial_usage = get_usage_stats()
        self.cost_limit = None

    def get_current_cost(self):
        current_usage = get_usage_stats()
        if not current_usage or not self.initial_usage:
            return 0.0

        input_tokens = current_usage.get('input_tokens', 0) - self.initial_usage.get('input_tokens', 0)
        output_tokens = current_usage.get('output_tokens', 0) - self.initial_usage.get('output_tokens', 0)
        pricing = self.pricing.get(self.model, {"input": 0.005, "output": 0.015})

        return (input_tokens / 1000) * pricing['input'] + (output_tokens / 1000) * pricing['output']

    def set_limit(self, limit_usd):
        self.cost_limit = limit_usd

    def check_limit(self):
        if self.cost_limit is None:
            return False
        current_cost = self.get_current_cost()
        if current_cost > self.cost_limit:
            print_error(f"Cost limit exceeded! Current: ${current_cost:.4f}, Limit: ${self.cost_limit:.4f}")
            return True
        return False

    def report(self):
        cost = self.get_current_cost()
        usage = get_usage_stats()
        initial = self.initial_usage

        print("\nCost Monitoring Report:")
        print(f"  Model: {self.model}")
        if usage and initial:
            input_delta = usage.get('input_tokens', 0) - initial.get('input_tokens', 0)
            output_delta = usage.get('output_tokens', 0) - initial.get('output_tokens', 0)
            print(f"  Input tokens: {input_delta:,}")
            print(f"  Output tokens: {output_delta:,}")
        print(f"  Estimated cost: ${cost:.4f}")
        if self.cost_limit:
            remaining = self.cost_limit - cost
            print(f"  Budget remaining: ${remaining:.4f}")

# Use cost monitor
monitor = CostMonitor(model="gpt-4o")
monitor.set_limit(0.10)
print("Cost monitor initialized with $0.10 limit")

for _ in range(2):
    qa(question="What is the meaning of life?")

monitor.report()
monitor.check_limit()

## Part 6: Best Practices

In [ ]:
print("1. Always enable tracking in development:")
print("   configure_dspy(lm=lm, track_usage=True)")
print()
print("2. Monitor usage during development:")
print("   - Identify expensive operations")
print("   - Compare different implementations")
print("   - Set cost budgets for testing")
print()
print("3. Optimize based on data:")
print("   - Use simpler modules when possible (Predict vs ChainOfThought)")
print("   - Reduce max_tokens for simple tasks")
print("   - Cache responses when appropriate")
print("   - Use cheaper models for simple operations")
print()
print("4. Production monitoring:")
print("   - Set up cost alerts")
print("   - Log usage per user/session")
print("   - Track trends over time")
print()
print("5. Cost optimization strategies:")
print("   - Use gpt-4o-mini for simple tasks")
print("   - Implement caching for repeated queries")
print("   - Batch similar requests")
print("   - Set appropriate max_tokens limits")

## Summary

**Key Takeaways:**
1. Enable usage tracking with `configure_dspy(track_usage=True)`
2. Use `get_usage_stats()` to retrieve current usage
3. Track usage deltas for module-level monitoring
4. Compare approaches to optimize for efficiency
5. Implement cost monitoring and alerts
6. Log usage data for production monitoring
7. Optimize based on real usage data